# slot-data-gen — demo

In [ ]:
import json
import logging
from pathlib import Path

from dotenv import load_dotenv

from slot_data_gen import GenConfig, GenJob, OpenAIModel, run_generation

logging.basicConfig(level=logging.INFO, format="%(name)s | %(message)s")
logging.getLogger("httpx").setLevel(logging.WARNING)
load_dotenv("../.env");

In [ ]:
# Configure the LLM

llm = OpenAIModel(model="deepseek-chat", api="chat")
llm_cfg = GenConfig(temperature=0.7, max_tokens=8192)

In [ ]:
# Define what to generate

job = GenJob(
    target_intent="BookHotel",
    slot_type_set=["city", "check_in_date", "n_guests", "room_type", "price_range"],
    out_path=Path("./out/book_hotel.json"),
    stage1_n_combos=5,
    stage2_n_samples_per_combo=10,
    filter_strategy="llm_judge",
)

In [ ]:
# Run the pipeline

samples = run_generation(
    llm=llm,
    llm_cfg=llm_cfg,
    job=job,
    judge_llm=llm,
    judge_cfg=llm_cfg,
)
print(f"\nGenerated {len(samples)} samples")

In [ ]:
# Inspect the output

records = json.loads(job.out_path.read_text())
print(f"Total: {len(records)} samples\n")

for i, record in enumerate(records[:3]):
    print(f"--- sample {i} ---")
    print(f"query: {record['query']}")
    for tok, tag in zip(record["tokens"], record["tags"]):
        marker = "  " if tag == "O" else "->"
        print(f"  {marker} {tok:<20} {tag}")
    print()